In [1]:
from ultralytics import YOLO
import cv2
import numpy as np
import os

os.chdir("/home/oriado/Files/meter_project")

In [2]:
model = YOLO("best.pt")

In [ ]:
def find_digit_strip(crop):
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape
    win = max(10, h // 7)
    margin = int(h * 0.15)
    search_start = margin
    search_end = h - margin - win
    best_y, best_mean = search_start, float("inf")
    for y in range(search_start, search_end):
        m = float(gray[y:y + win, :].mean())
        if m < best_mean:
            best_mean = m
            best_y = y
    pad = win // 3
    y1 = max(0, best_y - pad)
    y2 = min(h, best_y + win + pad)
    return crop[y1:y2, :]


def get_crop(image_path):
    img = cv2.imread(image_path)
    if img is None:
        return None, None

    results = model(img, verbose=False)
    boxes = results[0].boxes

    if len(boxes) == 0:
        return img, None

    best_idx = int(boxes.conf.argmax())
    coords = boxes.xyxy[best_idx].tolist()
    x1 = int(coords[0])
    y1 = int(coords[1])
    x2 = int(coords[2])
    y2 = int(coords[3])

    w = x2 - x1
    h = y2 - y1

    src = np.float32([[x1, y1], [x2, y1], [x2, y2], [x1, y2]])
    dst = np.float32([[0, 0], [w, 0], [w, h], [0, h]])
    M = cv2.getPerspectiveTransform(src, dst)
    crop = cv2.warpPerspective(img, M, (w, h))

    if crop.shape[0] > crop.shape[1] * 0.3:
        crop = find_digit_strip(crop)

    return img, crop


In [4]:
test_images = os.listdir("dataset/test/images")
sample_path = "dataset/test/images/" + test_images[0]

original, crop = get_crop(sample_path)

cv2.imwrite("example_original.jpg", original)
cv2.imwrite("example_crop.jpg", crop)

True

In [5]:
results = model(original, verbose=False)
img_with_box = results[0].plot()
cv2.imwrite("example_detection.jpg", img_with_box)

True

In [6]:
os.makedirs("examples/input", exist_ok=True)
os.makedirs("examples/output", exist_ok=True)

for i in range(min(5, len(test_images))):
    path = "dataset/test/images/" + test_images[i]
    orig, crop = get_crop(path)
    if crop is not None:
        cv2.imwrite(f"examples/input/test_{i}.jpg", orig)
        cv2.imwrite(f"examples/output/crop_{i}.jpg", crop)